# 134 · DeerFlow Embedded Client

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/134-deerflow-embedded-client/deerflow_embedded_client_workbook.ipynb)

**What this notebook teaches:**
- How DeerFlow exposes a three-endpoint HTTP API (upload, stream SSE, blocking chat)
- How to parse Server-Sent Events (SSE) in plain Python — no special library needed
- What each `data:` line looks like for each event type
- How `plan_mode=True` surfaces the planning step as visible `tool_call` events
- The key contrast between framework ownership models: LangGraph owns nothing, ADK owns the tool loop, DeerFlow owns skills + memory + tools

**Two halves:**
- **Part A (cells 1–7):** Pure Python — run anywhere, no server needed. Walk through the `DeerFlowClient` class and SSE parsing hands-on.
- **Part B (cells 8–10):** Requires a running DeerFlow instance. Fail-fast health check, then a real upload + stream + blocking chat demo.

| Approach | How you drive it | Runtime owns |
|----------|-----------------|-------------|
| LangGraph | `app.invoke(state)` — you build the graph | Nothing |
| Google ADK | `runner.run_async()` — ADK owns the tool loop | Tool-call loop |
| Agno | `agent.run()` — Agno owns memory + tools | Memory, tool registry |
| DeerFlow client | `client.stream()` → events | Skills, memory, tools, subagents |

In [ ]:
# Part A · Cell 1 — Install
%pip install -q httpx python-dotenv

## Part A · The DeerFlow HTTP API

DeerFlow exposes exactly three endpoints you need to know:

```
POST /api/files/upload          Upload a file into the thread workspace
                                Body: multipart/form-data
                                  file       = (filename, bytes, mime_type)
                                  thread_id  = your conversation key
                                Returns: { artifact_id: str }

POST /api/chat/stream           Send a message; receive a Server-Sent Event stream
                                Body: JSON
                                  message           = str
                                  thread_id         = str
                                  plan_mode         = bool   (default false)
                                  subagent_enabled  = bool   (default false)
                                Returns: text/event-stream

POST /api/chat                  Blocking variant of /stream
                                Same JSON body; waits for completion
                                Returns: { content: str }
```

The `thread_id` ties uploads, memory, and chat turns together.  
Every call in the same thread shares the same workspace and conversation history.

In [ ]:
# Part A · Cell 2 — Implement DeerFlowClient from scratch
# This mirrors src/workflow.py — reading the class here teaches you what the file does.

from __future__ import annotations

import json
from dataclasses import dataclass, field
from typing import Iterator

import httpx


@dataclass
class DeerFlowClient:
    """Thin HTTP wrapper for a running DeerFlow FastAPI service.

    DeerFlow owns skills, memory, and the LLM loop.
    You own only the HTTP request — that's the whole contract.
    """

    base_url: str
    thread_id: str
    _http: httpx.Client = field(
        default_factory=lambda: httpx.Client(
            timeout=httpx.Timeout(60.0, read=120.0)
        )
    )

    def upload(self, filename: str, content: str) -> str:
        """Attach a text file to the thread workspace; returns artifact_id."""
        resp = self._http.post(
            f"{self.base_url}/api/files/upload",
            files={"file": (filename, content.encode(), "text/markdown")},
            data={"thread_id": self.thread_id},
        )
        resp.raise_for_status()
        return resp.json().get("artifact_id", "unknown")

    def stream(
        self,
        message: str,
        *,
        plan_mode: bool = False,
        subagent_enabled: bool = False,
    ) -> Iterator[tuple[str, dict]]:
        """Yield (event_type, data) tuples from the SSE endpoint."""
        with self._http.stream(
            "POST",
            f"{self.base_url}/api/chat/stream",
            json={
                "message": message,
                "thread_id": self.thread_id,
                "plan_mode": plan_mode,
                "subagent_enabled": subagent_enabled,
            },
        ) as r:
            r.raise_for_status()
            for line in r.iter_lines():
                if not line.startswith("data:"):
                    continue
                raw = line.removeprefix("data:").strip()
                if raw == "[DONE]":
                    yield "end", {}
                    return
                try:
                    ev = json.loads(raw)
                    yield ev.get("type", "unknown"), ev
                except json.JSONDecodeError:
                    pass

    def chat(self, message: str, *, plan_mode: bool = False) -> str:
        """Blocking chat — joins message_chunk events into the final answer."""
        return "".join(
            d.get("content", "")
            for et, d in self.stream(message, plan_mode=plan_mode)
            if et == "message_chunk"
        )


print("DeerFlowClient defined.")
print(f"Methods: {[m for m in dir(DeerFlowClient) if not m.startswith('_')]}")

## Part A · SSE Event Types

Server-Sent Events arrive as plain text over HTTP chunked transfer.  
Each event is a `data:` line followed by a blank line.

DeerFlow emits four event types:

```
data: {"type": "message_chunk",  "content": "State graph is..."}           # LLM token
data: {"type": "tool_call",      "tool_name": "web_fetch", "args": {...}}   # tool being called
data: {"type": "tool_result",    "content": "<fetched page text>"}          # tool returned
data: {"type": "end",            "content": ""}                             # run finished
data: [DONE]                                                                  # SSE stream closed
```

The parser in `stream()` strips `data:`, skips non-data lines (comments, empty lines),  
handles `[DONE]` as the terminal signal, and JSON-decodes everything else.

**With `plan_mode=True`:**  
Before any `message_chunk` tokens arrive, you'll see a burst of `tool_call` events  
as DeerFlow surfaces its planning step. This lets you watch *what DeerFlow decides to do*  
before it starts executing — useful for debugging and understanding skill routing.

In [ ]:
# Part A · Cell 3 — Parse mock SSE events (no server required)
# Build a fake SSE stream as a list of raw lines, then run the same parser DeerFlowClient uses.

MOCK_SSE_LINES = [
    'data: {"type": "tool_call", "tool_name": "plan", "args": {"steps": ["read files", "summarise"]}}',
    'data: {"type": "message_chunk", "content": "State graph is an explicit"}',
    'data: {"type": "message_chunk", "content": " directed graph."}',
    'data: {"type": "tool_call", "tool_name": "web_fetch", "args": {"url": "https://example.com"}}',
    'data: {"type": "tool_result", "content": "<page content truncated>"}',
    'data: {"type": "message_chunk", "content": " Reactive loop runs until done."}',
    'data: {"type": "end", "content": ""}',
    'data: [DONE]',
]


def parse_sse_stream(lines: list[str]) -> list[tuple[str, dict]]:
    """Same logic as DeerFlowClient.stream() — runs on a list instead of HTTP."""
    events: list[tuple[str, dict]] = []
    for line in lines:
        if not line.startswith("data:"):
            continue
        raw = line.removeprefix("data:").strip()
        if raw == "[DONE]":
            events.append(("end", {}))
            break
        try:
            ev = json.loads(raw)
            events.append((ev.get("type", "unknown"), ev))
        except json.JSONDecodeError:
            pass
    return events


_ICONS = {"message_chunk": "·", "tool_call": "⚙", "tool_result": "✓", "end": "■"}

events = parse_sse_stream(MOCK_SSE_LINES)

print(f"Parsed {len(events)} events:\n")
for et, data in events:
    icon = _ICONS.get(et, "?")
    if et == "message_chunk":
        print(f"  {icon} [{et}]  content: {data.get('content', '')!r}")
    elif et == "tool_call":
        print(f"  {icon} [{et}]  tool: {data.get('tool_name', '?')}  args: {data.get('args', {})}")
    elif et == "tool_result":
        print(f"  {icon} [{et}]  content: {str(data.get('content', ''))[:60]!r}")
    else:
        print(f"  {icon} [{et}]")

## Part A · `plan_mode=True` — What Changes

Without `plan_mode`, DeerFlow goes straight to execution: you see `message_chunk` tokens  
and occasional `tool_call` / `tool_result` pairs interleaved as the agent works.

With `plan_mode=True`, DeerFlow surfaces an explicit planning step **before** execution:

```
plan_mode=False (default):
  · message_chunk    ← tokens start immediately
  ⚙ tool_call        ← tool calls appear mid-stream
  ✓ tool_result
  · message_chunk
  ■ end

plan_mode=True:
  ⚙ tool_call  tool=plan  ← planning step visible as a distinct event
  ✓ tool_result           ← plan confirmed / adjusted
  · message_chunk         ← execution begins
  ⚙ tool_call  tool=web_fetch
  ✓ tool_result
  · message_chunk
  ■ end
```

Use `plan_mode=True` when you want to:
- Debug skill routing (see exactly what DeerFlow plans to do)
- Build a UI that shows a progress indicator before the response starts
- Audit the agent's reasoning on a complex or multi-file task

In [ ]:
# Part A · Cell 4 — Simulate plan_mode=True event ordering
# Shows the difference in event sequence between the two modes.

MOCK_NO_PLAN = [
    'data: {"type": "message_chunk", "content": "State graph uses"}',
    'data: {"type": "message_chunk", "content": " directed edges."}',
    'data: {"type": "end", "content": ""}',
    'data: [DONE]',
]

MOCK_WITH_PLAN = [
    'data: {"type": "tool_call", "tool_name": "plan", "args": {"steps": ["read notes", "summarise three patterns"]}}',
    'data: {"type": "tool_result", "content": "Plan confirmed: 3 patterns to summarise."}',
    'data: {"type": "message_chunk", "content": "State graph uses"}',
    'data: {"type": "message_chunk", "content": " directed edges."}',
    'data: {"type": "end", "content": ""}',
    'data: [DONE]',
]


def describe_sequence(label: str, lines: list[str]) -> None:
    print(f"{label}:")
    for et, data in parse_sse_stream(lines):
        icon = _ICONS.get(et, "?")
        detail = ""
        if et == "tool_call":
            detail = f"  tool={data.get('tool_name', '?')}"
        elif et == "message_chunk":
            detail = f"  {data.get('content', '')!r}"
        elif et == "tool_result":
            detail = f"  {data.get('content', '')[:40]!r}"
        print(f"  {icon} {et}{detail}")
    print()


describe_sequence("plan_mode=False", MOCK_NO_PLAN)
describe_sequence("plan_mode=True", MOCK_WITH_PLAN)

In [ ]:
# Part A · Cell 5 — Event counter utility (used in Part B too)
# Counts events by type — same pattern as main.py's summary block.


def count_events(events: list[tuple[str, dict]]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for et, _ in events:
        counts[et] = counts.get(et, 0) + 1
    return counts


def join_text(events: list[tuple[str, dict]]) -> str:
    return "".join(
        data.get("content", "")
        for et, data in events
        if et == "message_chunk"
    )


# Verify on our mock streams
for label, lines in [("no-plan mock", MOCK_NO_PLAN), ("with-plan mock", MOCK_WITH_PLAN)]:
    evs = parse_sse_stream(lines)
    print(f"{label}:")
    print(f"  counts  : {count_events(evs)}")
    print(f"  text    : {join_text(evs)!r}")
    print()

print("Part A complete — all cells run without a server.")

---
## Part B · Live DeerFlow Run

The cells below require a running DeerFlow instance.

```bash
# 1. Clone and start DeerFlow (one-time setup — see runtime/README.md)
git clone https://github.com/bytedance/deer-flow ../deer-flow
cd ../deer-flow && pip install -e . && make dev

# 2. Optional: set a custom URL
export DEERFLOW_BASE_URL=http://localhost:8000
```

Cell 8 will raise immediately if DeerFlow is not reachable — **no silent hangs.**

In [ ]:
# Part B · Cell 6 — Connect (fail fast if DeerFlow is down)
import os

from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("DEERFLOW_BASE_URL", "http://localhost:8000")
THREAD_ID = "notebook-thread-134"

try:
    r = httpx.get(f"{BASE_URL}/api/health", timeout=5)
    r.raise_for_status()
    print(f"DeerFlow reachable at {BASE_URL}")
except Exception as exc:
    raise RuntimeError(
        f"DeerFlow is not reachable at {BASE_URL}.\n"
        "Start it with `make dev` in the deer-flow repo, then re-run this cell."
    ) from exc

client = DeerFlowClient(base_url=BASE_URL, thread_id=THREAD_ID)
print(f"Client ready  thread_id={THREAD_ID}")

In [ ]:
# Part B · Cell 7 — Upload a file, then stream with plan_mode=True

SAMPLE_CORPUS = """\
# Agent Framework Patterns

## State Graph
Explicit directed graph; nodes process state. Best for structured pipelines.

## Reactive Loop
Agent calls tools until done. Best for open-ended tasks and code generation.

## Multi-Agent
Supervisor delegates to specialists. Best for parallelisable research tasks.
"""

QUERY = "Summarise the three patterns in one sentence each."

print("Uploading course-notes.md ...")
artifact_id = client.upload("course-notes.md", SAMPLE_CORPUS)
print(f"  artifact_id: {artifact_id}")

print(f"\nStreaming (plan_mode=True) ...\nQ: {QUERY}\n")
live_events: list[tuple[str, dict]] = []
for et, data in client.stream(QUERY, plan_mode=True):
    live_events.append((et, data))
    icon = _ICONS.get(et, "?")
    if et == "message_chunk":
        print(data.get("content", ""), end="", flush=True)
    elif et == "tool_call":
        print(f"\n  {icon} tool_call: {data.get('tool_name', '?')}")
    elif et == "tool_result":
        print(f"  {icon} tool_result: {str(data.get('content', ''))[:60]!r}")
    elif et == "end":
        print(f"\n  {icon} end")

In [ ]:
# Part B · Cell 8 — Blocking chat call + event summary

QUERY2 = "Which pattern is best for a research report? One sentence."

print(f"Blocking chat ...\nQ: {QUERY2}")
answer = client.chat(QUERY2)
print(f"A: {answer}")

print("\n── Event summary (streaming run) ──")
counts = count_events(live_events)
for et, n in sorted(counts.items()):
    print(f"  {_ICONS.get(et, '?')} {et:<18} {n}x")

print(f"\nThread : {THREAD_ID}")
print(f"Server : {BASE_URL}")